In [1]:
# Make the notebook behave as if it were running from the repo root.
# evaluation.py uses a relative data path, so this matters.
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import time
import numpy as np
import pandas as pd

from data_loading import feature_groups
from evaluation import SEED, N_FOLDS, gini_normalized, cross_validate_model, load_train
from train_baseline_models import _build_preprocessor

train = load_train()  # only the 80% split -- load_final_test is never called in this notebook
categorical_cols, quantity_cols = feature_groups(train.columns)

# Same 37-column set as #13's calc-dropped model: all ps_calc_* columns removed.
cat_nocalc = [c for c in categorical_cols if "_calc_" not in c]
qty_nocalc = [c for c in quantity_cols if "_calc_" not in c]

print(f"rows {len(train):,} | 37-column set: cat {len(cat_nocalc)}, qty {len(qty_nocalc)}, "
      f"total {len(cat_nocalc) + len(qty_nocalc)}")


rows 475,967 | 37-column set: cat 25, qty 12, total 37


In [2]:
# Step 1 of the ticket's design: re-derive the individually-dead columns
# by rerunning #13's permutation_importance step fresh, rather than copying
# its column list -- #13's own writeup flags that a rerun landed on 11
# columns instead of 12, right at the noise floor around a zero threshold.
#
# Identical method to notebooks/02_feature_importance_review.ipynb cell 7:
# same 37-column forest (#9's RF settings), same fold-5 holdout from the
# harness's own 5-fold split, same permutation_importance call.
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

X_nc = train[cat_nocalc + qty_nocalc]
y = train["target"]
Xa, ya = X_nc.values, y.values


def forest_37col():
    """Identical to the #9 baseline forest -- only the column count differs."""
    return make_pipeline(
        _build_preprocessor(len(cat_nocalc), len(qty_nocalc), scale=False),
        RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                               n_jobs=-1, random_state=SEED),
    )


splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
tr_idx, va_idx = list(splitter.split(Xa, ya))[-1]
print(f"fit on {len(tr_idx):,} rows, permute on {len(va_idx):,} rows")

t0 = time.perf_counter()
perm_model = forest_37col()
perm_model.fit(Xa[tr_idx], ya[tr_idx])

baseline_gini = gini_normalized(ya[va_idx], perm_model.predict_proba(Xa[va_idx])[:, 1])
print(f"baseline gini on that fold: {baseline_gini:.5f}")


def gini_scorer(estimator, X, y):
    return gini_normalized(y, estimator.predict_proba(X)[:, 1])


result = permutation_importance(
    perm_model, Xa[va_idx], ya[va_idx],
    scoring=gini_scorer, n_repeats=3, random_state=SEED, n_jobs=-2,
)
perm_runtime = time.perf_counter() - t0
print(f"permutation importance took {perm_runtime/60:.1f} min for {Xa.shape[1]} columns")

perm = (pd.DataFrame({"column": X_nc.columns,
                      "gini_drop": result.importances_mean,
                      "drop_std": result.importances_std})
          .sort_values("gini_drop", ascending=False)
          .reset_index(drop=True))
perm.index += 1

n_dead = int((perm.gini_drop <= 0).sum())
print(f"\ncolumns whose shuffling did NOT hurt (drop <= 0): {n_dead} of {len(perm)}\n")
pd.set_option("display.max_rows", None)
perm


fit on 380,774 rows, permute on 95,193 rows


baseline gini on that fold: 0.27667


permutation importance took 1.0 min for 37 columns

columns whose shuffling did NOT hurt (drop <= 0): 11 of 37



,column,gini_drop,drop_std
1,ps_ind_05_cat,0.028108,0.002041
2,ps_ind_17_bin,0.013593,0.001105
3,ps_ind_03,0.013198,0.001099
4,ps_car_13,0.012754,0.001825
5,ps_reg_01,0.008532,0.001182
6,ps_reg_02,0.008057,0.001719
7,ps_reg_03,0.007408,0.001244
8,ps_car_07_cat,0.007223,0.001257
9,ps_ind_15,0.004723,0.002020
10,ps_car_04_cat,0.003912,0.000605


In [3]:
# The dead-column list, fresh off this run -- not copied from #13.
dead_cols = perm.loc[perm.gini_drop <= 0, "column"].tolist()
print(f"{len(dead_cols)} individually-dead columns (permutation drop <= 0):")
for c in sorted(dead_cols):
    row = perm.loc[perm.column == c].iloc[0]
    print(f"  {c:16} drop {row.gini_drop:+.6f}")

cat_dropped = [c for c in cat_nocalc if c not in dead_cols]
qty_dropped = [c for c in qty_nocalc if c not in dead_cols]
print(f"\nremaining columns: {len(cat_dropped) + len(qty_dropped)} "
      f"(cat {len(cat_dropped)}, qty {len(qty_dropped)})")


11 individually-dead columns (permutation drop <= 0):
  ps_car_02_cat    drop -0.000335
  ps_car_05_cat    drop -0.001299
  ps_car_08_cat    drop -0.000124
  ps_car_12        drop -0.002229
  ps_car_14        drop -0.000170
  ps_ind_10_bin    drop +0.000000
  ps_ind_11_bin    drop -0.000042
  ps_ind_12_bin    drop -0.000052
  ps_ind_13_bin    drop +0.000000
  ps_ind_14        drop -0.000187
  ps_ind_18_bin    drop -0.000413

remaining columns: 26 (cat 17, qty 9)


In [4]:
# Score the reduced model -- #8's cross-validation runner, everything else
# unchanged from #9's ensemble (same RF hyperparameters, same preprocessing
# shape, same seed and folds). One change from the 37-column reference:
# the dead columns are gone.
def forest_dropped():
    return make_pipeline(
        _build_preprocessor(len(cat_dropped), len(qty_dropped), scale=False),
        RandomForestClassifier(n_estimators=200, min_samples_leaf=50,
                               n_jobs=-1, random_state=SEED),
    )


X_dropped = train[cat_dropped + qty_dropped]

t0 = time.perf_counter()
dropped_mean, dropped_std, dropped_scores = cross_validate_model(forest_dropped(), X_dropped, y)
cv_runtime = time.perf_counter() - t0
print(f"\nGini  {dropped_mean:+.5f} +/- {dropped_std:.5f}")
print(f"Runtime  {cv_runtime/60:.1f} min (plus {perm_runtime/60:.1f} min for permutation importance)")


      fold 1  gini +0.26651  (claims in fold: 3,492)


      fold 2  gini +0.28004  (claims in fold: 3,493)


      fold 3  gini +0.27150  (claims in fold: 3,492)


      fold 4  gini +0.27415  (claims in fold: 3,492)


      fold 5  gini +0.28568  (claims in fold: 3,492)
      mean +0.27558  std 0.00667

Gini  +0.27558 +/- 0.00667
Runtime  1.3 min (plus 1.0 min for permutation importance)


In [5]:
# Two comparisons, per the ticket's ACs:
#  - delta vs the logistic regression baseline (results.md's standard column,
#    and the "Beats std?" verdict is always measured against this delta)
#  - gain/loss vs the 37-column calc-dropped reference this ticket started
#    from (results.md: 0.27244 +/- 0.00305) -- this is what decides keep/reject

LR_MEAN = 0.2572
REFERENCE_MEAN, REFERENCE_STD = 0.27244, 0.00305

delta_lr = dropped_mean - LR_MEAN
beats_std_lr = "yes" if delta_lr > dropped_std else "no"

delta_ref = dropped_mean - REFERENCE_MEAN
keep = delta_ref > dropped_std
decision = "KEEP" if keep else "REJECT"

print(f"vs logistic regression baseline ({LR_MEAN:.4f}): {delta_lr:+.5f}")
print(f"Beats std? {beats_std_lr}  (row std {dropped_std:.5f})")
print()
print(f"vs 37-column calc-dropped reference ({REFERENCE_MEAN:.5f} +/- {REFERENCE_STD:.5f}): "
      f"{delta_ref:+.5f}")
print(f"gain exceeds this row's std ({dropped_std:.5f})? {keep}")
print(f"\nDecision: {decision}")


vs logistic regression baseline (0.2572): +0.01838
Beats std? yes  (row std 0.00667)

vs 37-column calc-dropped reference (0.27244 +/- 0.00305): +0.00314
gain exceeds this row's std (0.00667)? False

Decision: REJECT
